# 🧩 LEGO Vision AI — Pipeline v6.0 (Lightning AI)\n\n**Arquitectura Híbrida**: Imágenes generadas localmente (Mac M4) → Entrenamiento en Cloud (GPU)\n\n| Celda | Fase | Descripción |\n|-------|------|-------------|\n| C0 | Setup | Configuración del entorno Lightning AI |\n| C1 | Dataset | Descomprimir dataset pre-renderizado |\n| C2 | Train | Entrenamiento YOLO (Universal Detector) |\n| C3 | Index | Generación de índice vectorial (FAISS) |\n| C4 | Sync | Empaquetado y sincronización de resultados |\n

In [ ]:
# =================================================================\n
# CELDA 0: Setup Lightning AI (v6.0 — Hybrid Pipeline)\n
# =================================================================\n
import os, sys, json, shutil, logging, zipfile, urllib.request\n
\n
# --- Find PROJECT_ROOT ---\n
PROJECT_ROOT = ''\n
for parent in [os.getcwd()] + [os.path.dirname(os.getcwd())]:\n
    if os.path.isdir(os.path.join(parent, 'src')):\n
        PROJECT_ROOT = parent\n
        break\n
if not PROJECT_ROOT:\n
    PROJECT_ROOT = os.getcwd()\n
\n
WORKSPACE_DIR = os.path.join(os.getcwd(), 'lightning_workspace')\n
DATASET_DIR = os.path.join(WORKSPACE_DIR, 'datasets')\n
os.makedirs(WORKSPACE_DIR, exist_ok=True)\n
os.makedirs(DATASET_DIR, exist_ok=True)\n
\n
sys.path.insert(0, PROJECT_ROOT)\n
\n
# --- Install dependencies ---\n
os.system('pip install ultralytics pandas requests scikit-learn psutil pyyaml faiss-cpu google-api-python-client google-auth-oauthlib -q')\n
\n
# --- Logging ---\n
LOG_FILE_PATH = os.path.join(WORKSPACE_DIR, 'pipeline_exec.log')\n
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',\n
    handlers=[logging.FileHandler(LOG_FILE_PATH), logging.StreamHandler()])\n
logger = logging.getLogger('LegoVision')\n
\n
# --- Load Launch Config ---\n
LAUNCH_CONFIG = {}\n
config_path = os.path.join(PROJECT_ROOT, 'config_train.json')\n
if os.path.exists(config_path):\n
    with open(config_path, 'r') as f:\n
        LAUNCH_CONFIG = json.load(f)\n
    logger.info(f'📋 Config loaded: {LAUNCH_CONFIG.get("session_reference", "unknown")}')\n
\n
RENDER_ENGINE = 'CYCLES'  # Informational only (rendering was done locally)\n
SET_ID = LAUNCH_CONFIG.get('session_reference', 'custom').split('_')[0] if LAUNCH_CONFIG else 'custom'\n
\n
# --- Pipeline Timer (optional) ---\n
try:\n
    from src.utils.pipeline_timer import PipelineTimer\n
    timer = PipelineTimer()\n
    timer.detect_hardware()\n
except ImportError:\n
    class DummyTimer:\n
        def detect_hardware(self): pass\n
        def log_config(self, c): pass\n
        def step(self, name):\n
            class Ctx:\n
                def __enter__(s): return s\n
                def __exit__(s, *a): pass\n
            return Ctx()\n
        def save_report(self, p): pass\n
    timer = DummyTimer()\n
\n
# --- Drive Credentials (copy if present) ---\n
for cred_file in ['token_1973.pickle', 'credentials.json']:\n
    src = os.path.join(PROJECT_ROOT, cred_file)\n
    dst = os.path.join(WORKSPACE_DIR, cred_file)\n
    if os.path.exists(src) and not os.path.exists(dst):\n
        shutil.copy(src, dst)\n
\n
logger.info(f'Pipeline v6.0 (Hybrid) | Root: {PROJECT_ROOT} | Workspace: {WORKSPACE_DIR}')\n


In [ ]:
# =================================================================\n
# CELDA 1: 📦 Cargar Dataset Pre-Renderizado (Local M4)\n
# =================================================================\n
import os, zipfile, json, glob\n
\n
logger.info('>>> FASE 1/4: Carga del Dataset Pre-Renderizado')\n
\n
# Strategy: Find the dataset ZIP in the project root or workspace\n
zip_candidates = []\n
for search_dir in [PROJECT_ROOT, WORKSPACE_DIR, os.getcwd()]:\n
    zip_candidates.extend(glob.glob(os.path.join(search_dir, 'lightning_dataset_*.zip')))\n
\n
if not zip_candidates:\n
    raise FileNotFoundError('❌ No se encontró ningún lightning_dataset_*.zip. Genera el dataset localmente primero.')\n
\n
# Use the most recent one\n
dataset_zip = max(zip_candidates, key=os.path.getmtime)\n
logger.info(f'📦 Dataset encontrado: {os.path.basename(dataset_zip)}')\n
\n
# Extract to workspace\n
extract_dir = WORKSPACE_DIR\n
with zipfile.ZipFile(dataset_zip, 'r') as z:\n
    z.extractall(extract_dir)\n
    logger.info(f'✅ Extraído en {extract_dir}')\n
\n
# Find render_local directory (it should be inside the zip)\n
RENDER_LOCAL = os.path.join(extract_dir, 'render_local')\n
if not os.path.exists(RENDER_LOCAL):\n
    # Maybe extracted directly\n
    for d in os.listdir(extract_dir):\n
        if os.path.isdir(os.path.join(extract_dir, d)) and d != 'src':\n
            RENDER_LOCAL = os.path.join(extract_dir, d)\n
            break\n
\n
# Read manifest\n
manifest_path = os.path.join(RENDER_LOCAL, 'dataset_manifest.json')\n
if os.path.exists(manifest_path):\n
    with open(manifest_path, 'r') as f:\n
        manifest = json.load(f)\n
    print(f'📋 Manifest: {manifest["total_pieces"]} piezas | {manifest["total_images"]} imágenes')\n
    for p in manifest.get('pieces', []):\n
        print(f'   • {p["piece_id"]}: {p["images"]} imgs, {p["labels"]} labels')\n
else:\n
    logger.warning('⚠️ No se encontró dataset_manifest.json')\n
\n
# Build unified dataset by merging all piece subfolders\n
UNIFIED_DATASET = os.path.join(DATASET_DIR, SET_ID)\n
unified_images = os.path.join(UNIFIED_DATASET, 'images')\n
unified_labels = os.path.join(UNIFIED_DATASET, 'labels')\n
os.makedirs(unified_images, exist_ok=True)\n
os.makedirs(unified_labels, exist_ok=True)\n
\n
import shutil\n
total_merged = 0\n
for piece_dir in sorted(os.listdir(RENDER_LOCAL)):\n
    piece_path = os.path.join(RENDER_LOCAL, piece_dir)\n
    if not os.path.isdir(piece_path): continue\n
    img_dir = os.path.join(piece_path, 'images')\n
    lbl_dir = os.path.join(piece_path, 'labels')\n
    if not os.path.exists(img_dir): continue\n
    \n
    for img in os.listdir(img_dir):\n
        # Prefix with piece_id to avoid collisions\n
        new_name = f'{piece_dir}_{img}'\n
        shutil.copy2(os.path.join(img_dir, img), os.path.join(unified_images, new_name))\n
        # Copy matching label\n
        lbl_name = img.replace('.jpg', '.txt').replace('.png', '.txt')\n
        lbl_path = os.path.join(lbl_dir, lbl_name)\n
        if os.path.exists(lbl_path):\n
            shutil.copy2(lbl_path, os.path.join(unified_labels, f'{piece_dir}_{lbl_name}'))\n
        total_merged += 1\n
\n
# Generate unified data.yaml\n
data_yaml = os.path.join(UNIFIED_DATASET, 'data.yaml')\n
with open(data_yaml, 'w') as f:\n
    f.write(f'path: {UNIFIED_DATASET}\\n')\n
    f.write('train: images\\n')\n
    f.write('val: images\\n')\n
    f.write('nc: 1\\n')\n
    f.write("names: ['lego']\\n")\n
\n
logger.info(f'✅ Dataset unificado: {total_merged} imágenes en {UNIFIED_DATASET}')\n
# Make PARTS_TO_TRAIN available for downstream cells\n
PARTS_TO_TRAIN = [{'ldraw_id': p['piece_id'], 'name': p['piece_id']} for p in manifest.get('pieces', [])] if 'manifest' in dir() else [{'ldraw_id': SET_ID, 'name': SET_ID}]\n


In [ ]:
# =================================================================\n
# CELDA 2: 🏋️ YOLO Training (Universal Detector)\n
# =================================================================\n
from ultralytics import YOLO\n
import torch, time as _time, glob, shutil, datetime\n
\n
logger.info('>>> FASE 2/4: Entrenamiento YOLO')\n
\n
dataset_path = os.path.join(DATASET_DIR, SET_ID)\n
data_yaml = os.path.join(dataset_path, 'data.yaml')\n
\n
if not os.path.exists(data_yaml):\n
    raise FileNotFoundError(f'❌ No se encontró data.yaml en {dataset_path}')\n
\n
results_dir = os.path.join(WORKSPACE_DIR, 'results')\n
\n
# Progressive Training: use latest weights if available\n
base_model_path = 'yolo11s-seg.pt'\n
models_dir = os.path.join(PROJECT_ROOT, 'models', 'yolo_model')\n
os.makedirs(models_dir, exist_ok=True)\n
\n
trained_weights = sorted(glob.glob(os.path.join(models_dir, 'universal_detector_*.pt')))\n
if trained_weights:\n
    base_model_path = trained_weights[-1]\n
    print(f'🧠 Incremental training from: {os.path.basename(base_model_path)}')\n
else:\n
    print('🌱 Training from base (yolo11s-seg.pt)')\n
\n
model = YOLO(base_model_path, task='segment')\n
\n
num_gpus = torch.cuda.device_count()\n
train_device = list(range(num_gpus)) if num_gpus > 1 else (0 if num_gpus == 1 else 'cpu')\n
\n
if torch.cuda.is_available():\n
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'\n
    torch.cuda.empty_cache()\n
\n
# Callback for epoch logging\n
_t_epoch = [_time.time()]\n
def _on_epoch_end(trainer):\n
    elapsed = _time.time() - _t_epoch[0]\n
    m = trainer.metrics or {}\n
    vram = torch.cuda.memory_reserved(0)/1e9 if torch.cuda.is_available() else 0\n
    print(f'[EPOCH {trainer.epoch+1}/{trainer.epochs}] {elapsed:.1f}s | mAP50: {m.get("metrics/mAP50(B)",0):.4f} | VRAM: {vram:.1f}GB', flush=True)\n
model.add_callback('on_fit_epoch_end', _on_epoch_end)\n
\n
TRAIN_EPOCHS = 150\n
print(f'🚀 Training on: {train_device} | Epochs: {TRAIN_EPOCHS}')\n
\n
with timer.step('YOLO Training'):\n
    model.train(\n
        task='segment',\n
        data=data_yaml,\n
        epochs=TRAIN_EPOCHS,\n
        patience=10,\n
        imgsz=640,\n
        project=results_dir,\n
        name=f'yolo11_{SET_ID}',\n
        batch=-1,\n
        device=train_device,\n
        workers=4,\n
        cache=True,\n
        amp=True,\n
        optimizer='auto',\n
        verbose=False,\n
        mosaic=0.7,\n
        mixup=0.1,\n
        scale=0.05,\n
        degrees=180.0,\n
        translate=0.2,\n
        hsv_s=0.7,\n
        hsv_v=0.4,\n
        bgr=0.2,\n
        blur=0.1,\n
        erasing=0.1\n
    )\n
\n
# Auto-version the model\n
dirs = glob.glob(os.path.join(results_dir, f'yolo11_{SET_ID}*'))\n
latest_dir = max(dirs, key=os.path.getmtime) if dirs else ''\n
best_pt = os.path.join(latest_dir, 'weights', 'best.pt') if latest_dir else ''\n
if os.path.exists(best_pt):\n
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')\n
    dest = os.path.join(models_dir, f'universal_detector_{ts}.pt')\n
    shutil.copy2(best_pt, dest)\n
    print(f'✅ Model saved: {dest}')\n
else:\n
    print('⚠️ No best.pt found')\n
